# 06 — Model Evaluation

## Objectives
- Compare five models using R², MAE, RMSE, MAPE, and MdAPE.
- Analyze prediction errors across sale-price bands.
- Export results to `metrics_summary.csv`.

## Models
- Linear Regression (No FE)
- Decision Tree
- Random Forest
- XGBoost
- LightGBM

The linear regression baseline uses the original preprocessed features.
The other four models use engineered features. All five models were
trained on log-transformed sale prices. Dollar and percentage errors
are evaluated on the original price scale.

The evaluation uses the April 2026 test set.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_DIR = (
    Path.home() / "Downloads" / "IDX-Exchange-Data-Science"
)

# Load predictions exported from the previous notebooks.
baseline = pd.read_csv(
    PROJECT_DIR / "baseline_test_predictions.csv"
)
trees = pd.read_csv(
    PROJECT_DIR / "tree_test_predictions.csv"
)
boosting = pd.read_csv(
    PROJECT_DIR / "boosting_test_predictions.csv"
)

# Sort by test-row number and verify unique row identifiers.
prediction_files = {
    "Baseline": baseline,
    "Trees": trees,
    "Boosting": boosting,
}

for name, data in prediction_files.items():
    assert data["TestRow"].is_unique, (
        f"Duplicate test rows in {name}."
    )
    data.sort_values("TestRow", inplace=True)
    data.reset_index(drop=True, inplace=True)

# Verify matching row numbers and actual prices.
for name, data in [("Trees", trees), ("Boosting", boosting)]:
    assert np.array_equal(
        baseline["TestRow"].to_numpy(),
        data["TestRow"].to_numpy(),
    ), f"Test rows do not match: {name}."

    assert np.allclose(
        baseline["ActualPrice"].to_numpy(),
        data["ActualPrice"].to_numpy(),
        rtol=0,
        atol=0.01,
    ), f"Actual prices do not match: {name}."

# Combine predictions after checking alignment.
predictions = baseline.copy()

for data in [trees, boosting]:
    model_columns = data.columns.difference(
        ["TestRow", "ActualPrice"], sort=False
    )
    predictions[model_columns] = data[model_columns]

model_names = [
    "Linear Regression (No FE)",
    "Decision Tree",
    "Random Forest",
    "XGBoost",
    "LightGBM",
]

assert len(predictions) > 0
assert np.isfinite(
    predictions[["ActualPrice"] + model_names].to_numpy()
).all()
assert predictions["ActualPrice"].gt(0).all()
assert predictions[model_names].gt(0).all().all()

print("Test records:", len(predictions))
print("Models:", len(model_names))
print("Test-row and actual-price checks passed.")

display(predictions.head())

Test records: 11684
Models: 5
Test-row and actual-price checks passed.


,TestRow,ActualPrice,Linear Regression (No FE),Decision Tree,Random Forest,XGBoost,LightGBM
0,0,1750000.0,1.488868e+06,1.684823e+06,1.496095e+06,1426361.10,1.488791e+06
1,1,600000.0,7.291163e+05,5.995745e+05,7.788199e+05,736016.94,6.981907e+05
2,2,3625000.0,3.223221e+06,3.425977e+06,3.666507e+06,3787792.80,3.674566e+06
3,3,1450000.0,1.007280e+06,1.377313e+06,1.112512e+06,1126550.00,1.073006e+06
4,4,1500000.0,1.340455e+06,1.663001e+06,1.615056e+06,1498023.20,1.346214e+06


In [2]:
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error,
)

actual = predictions["ActualPrice"].to_numpy()

metric_rows = []

for model_name in model_names:
    predicted = predictions[model_name].to_numpy()

    # Calculate absolute percentage errors on the original price scale.
    absolute_percentage_errors = (
        np.abs(actual - predicted) / actual * 100
    )

    metric_rows.append({
        "Model": model_name,
        "Feature Engineering": (
            "No" if model_name == "Linear Regression (No FE)"
            else "Yes"
        ),
        "N": len(actual),
        "R² (price)": r2_score(actual, predicted),
        "R² (log price)": r2_score(
            np.log(actual), np.log(predicted)
        ),
        "MAE ($)": mean_absolute_error(actual, predicted),
        "RMSE ($)": np.sqrt(
            mean_squared_error(actual, predicted)
        ),
        "MAPE (%)": np.mean(absolute_percentage_errors),
        "MdAPE (%)": np.median(absolute_percentage_errors),
    })

overall_metrics = pd.DataFrame(metric_rows)

display(
    overall_metrics
    .sort_values("MdAPE (%)")
    .style.hide(axis="index")
    .format({
        "N": "{:,}",
        "R² (price)": "{:.4f}",
        "R² (log price)": "{:.4f}",
        "MAE ($)": "{:,.2f}",
        "RMSE ($)": "{:,.2f}",
        "MAPE (%)": "{:.2f}",
        "MdAPE (%)": "{:.2f}",
    })
)

Model,Feature Engineering,N,R² (price),R² (log price),MAE ($),RMSE ($),MAPE (%),MdAPE (%)
XGBoost,Yes,"11,684",0.8355,0.9343,"162,515.54","442,508.84",11.61,8.08
LightGBM,Yes,"11,684",0.8361,0.9332,"163,906.42","441,673.26",11.76,8.27
Linear Regression (No FE),No,"11,684",0.8256,0.9260,"172,630.73","455,641.17",12.45,8.79
Random Forest,Yes,"11,684",0.7707,0.9094,"192,876.20","522,483.31",13.40,9.04
Decision Tree,Yes,"11,684",0.7479,0.8953,"204,703.81","547,840.88",14.38,9.52


XGBoost achieved the lowest MdAPE at 8.08%, indicating that at least
half of the test properties had an absolute percentage error of
8.08% or less. Across all five models, MAPE exceeded MdAPE, suggesting
that larger percentage errors for some properties raised the mean
above the median.

In [4]:
# Define price bands using actual sale prices.
price_bins = [
    0,
    500_000,
    1_000_000,
    2_000_000,
    np.inf,
]


price_labels = [
    "Under $500k",
    "$500k to $1M",
    "$1M to $2M",
    "$2M+",
]

predictions["PriceBand"] = pd.cut(
    predictions["ActualPrice"],
    bins=price_bins,
    labels=price_labels,
    right=False,
)

band_rows = []

for price_band in price_labels:
    group = predictions.loc[
        predictions["PriceBand"].eq(price_band)
    ]

    if group.empty:
        continue

    actual_band = group["ActualPrice"].to_numpy()

    for model_name in model_names:
        predicted_band = group[model_name].to_numpy()

        absolute_percentage_errors = (
            np.abs(actual_band - predicted_band)
            / actual_band
            * 100
        )

        band_rows.append({
            "Price Band": price_band,
            "Model": model_name,
            "N": len(group),
            "MAE ($)": mean_absolute_error(
                actual_band, predicted_band
            ),
            "RMSE ($)": np.sqrt(
                mean_squared_error(
                    actual_band, predicted_band
                )
            ),
            "MAPE (%)": np.mean(absolute_percentage_errors),
            "MdAPE (%)": np.median(absolute_percentage_errors),
        })

price_band_metrics = pd.DataFrame(band_rows)

display(
    price_band_metrics.style
    .hide(axis="index")
    .format({
        "N": "{:,}",
        "MAE ($)": "{:,.2f}",
        "RMSE ($)": "{:,.2f}",
        "MAPE (%)": "{:.2f}",
        "MdAPE (%)": "{:.2f}",
    })
)

Price Band,Model,N,MAE ($),RMSE ($),MAPE (%),MdAPE (%)
Under $500k,Linear Regression (No FE),"1,621","53,353.39","87,496.57",16.64,9.12
Under $500k,Decision Tree,"1,621","61,278.63","100,082.38",18.67,9.98
Under $500k,Random Forest,"1,621","57,992.27","97,171.44",17.94,8.52
Under $500k,XGBoost,"1,621","51,683.84","83,613.59",15.85,8.62
Under $500k,LightGBM,"1,621","51,689.30","84,866.49",15.90,8.66
$500k to $1M,Linear Regression (No FE),"4,895","74,509.19","108,357.79",10.09,7.30
$500k to $1M,Decision Tree,"4,895","81,801.18","126,080.70",11.03,7.31
$500k to $1M,Random Forest,"4,895","77,549.73","116,348.79",10.55,6.95
$500k to $1M,XGBoost,"4,895","67,807.25","100,265.09",9.16,6.46
$500k to $1M,LightGBM,"4,895","69,124.15","101,941.70",9.33,6.68


### Performance Across Price Bands

All five models achieved their lowest MAPE and MdAPE in the
$500k–$1M price band. For XGBoost, MAPE was 9.16% and MdAPE was
6.46% in this segment.

Properties priced at $2M or above had the highest MdAPE across
all models, indicating greater typical relative errors. LightGBM
achieved the lowest MAPE in this band at 15.38%, while XGBoost
had the lowest MdAPE at 12.68%.

Below $500k, Random Forest achieved the lowest MdAPE at 8.52%,
while XGBoost achieved the lowest MAPE at 15.85%. This illustrates
how model rankings can differ between median and mean errors.

In [5]:
# Combine overall and price-band metrics in one summary file.
overall_export = overall_metrics.copy()
overall_export.insert(0, "Price Band", "All prices")
overall_export.insert(0, "Scope", "Overall")

band_export = price_band_metrics.copy()
band_export.insert(0, "Scope", "Price band")
band_export["Feature Engineering"] = np.where(
    band_export["Model"].eq("Linear Regression (No FE)"),
    "No",
    "Yes",
)

# Log-price and price R² are reported for the overall evaluation only.
metrics_summary = pd.concat(
    [overall_export, band_export],
    ignore_index=True,
)

output_path = PROJECT_DIR / "metrics_summary.csv"
metrics_summary.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Summary rows:", len(metrics_summary))

Saved: /Users/jasmineyuan/Downloads/IDX-Exchange-Data-Science/metrics_summary.csv
Summary rows: 25
